# BO benchmark on the BOCoDe suite
Same models, acquisition functions, metrics and report as the source-model notebook, on the 13 constrained engineering problems. One `report.html` per problem. The seven CEC2020 ids in `bo_bench/problems/bocode_problem.py` are a guess; set them to the ones your partner used.

In [ ]:
# 1. Setup: Drive, repo, packages. Safe to re-run after a disconnect.
REPO_URL = "https://github.com/josefullrich/ASML.git"   # private: add a GITHUB_TOKEN secret in Colab
BRANCH   = "main"
DRIVE    = "/content/drive/MyDrive/ASML"                 # holds data/ and results/ (not in git)

import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")
def secret(name):
    try: return userdata.get(name)
    except Exception: return None
tok = secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
if not os.path.isdir("/content/ASML"):
    subprocess.run(["git", "clone", "-b", BRANCH, url, "/content/ASML"], check=True)
else:
    subprocess.run(["git", "-C", "/content/ASML", "pull"], check=False)
if not os.path.isdir("/content/tfm_onnx_in_matlab"):   # weighted classifier graphs
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/iwoodsawyer/tfm_onnx_in_matlab.git", "/content/tfm_onnx_in_matlab"], check=True)
!pip -q install botorch bocode causilo structured-data-models plotly onnx onnxscript statsmodels
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
os.makedirs(f"{DRIVE}/results/bo_bench", exist_ok=True)
%cd /content/ASML/notebooks
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (the TFMs will be very slow)")

In [ ]:
# 2. Settings. Anything in bo_bench/config.py can go here.
import json
SETTINGS = dict(
    results_dir = f"{DRIVE}/results/bo_bench",
    data_dir    = f"{DRIVE}/data",
    srcmdl_dir  = "/content/ASML/src",          # folder containing srcmdl.py
    ivo_repo    = "/content/tfm_onnx_in_matlab",
    budget_hours = 11.0,                        # stops cleanly and saves; run the cell again to continue
    models = ["kumo", "causilo"],
    kumo_size = "large",
    n_estimators = 1,
    bocode_seeds = [0, 1, 2],
    bocode_n_iter = 300,
    bocode_noise = 0.0,                         # relative Gaussian noise on every output; >0 for non-deterministic tests
    n_candidates = 4096,
)
json.dump(SETTINGS, open("/content/bo_bench_settings.json", "w"), indent=1)
print(json.dumps(SETTINGS, indent=1))

In [ ]:
!python -m bo_bench.run estimate --config /content/bo_bench_settings.json --set problem=bocode:WeldedBeamSO

**Run the whole suite.** One time budget for the session; re-run to resume.

In [ ]:
!python -m bo_bench.run all --suite --config /content/bo_bench_settings.json

In [ ]:
!python -m bo_bench.run report --suite --config /content/bo_bench_settings.json
import glob; print("\n".join(sorted(glob.glob(f"{DRIVE}/results/bo_bench/bocode_*/report.html"))))